In [36]:
import pandas as pd
from pathlib import Path

print("Notebook is working")

Notebook is working


In [37]:
import holidays

hours = pd.date_range(
    start="2022-01-01",
    end="2026-01-01",
    freq="h",
    tz="UTC",
    inclusive="left"
)

calendar = pd.DataFrame({"timestamp_utc": hours})
local_time = calendar["timestamp_utc"].dt.tz_convert("Europe/Rome")

calendar["hour_local"] = local_time.dt.hour
calendar["day_of_week"] = local_time.dt.dayofweek
calendar["month"] = local_time.dt.month
calendar["year"] = local_time.dt.year



In [38]:
calendar["is_weekend"] = (
    calendar["day_of_week"] >= 5
).astype(int)



In [39]:
italian_holidays = holidays.Italy(
    years=calendar["year"].unique().tolist()
)



In [40]:
calendar["is_holiday"] = local_time.dt.date.map(
    lambda date: date in italian_holidays
).astype(int)



In [41]:
# Exclude the National Unity Day Sunday celebration
unity_sunday = (
    (local_time.dt.month == 11)
    & (local_time.dt.day <= 7)
    & (local_time.dt.dayofweek == 6)
    & (local_time.dt.day != 1)
)

calendar.loc[unity_sunday, "is_holiday"] = 0

In [42]:
calendar["is_summer"] = (
    calendar["month"].isin([6, 7, 8])
).astype(int)



In [43]:
calendar["is_christmas_period"] = (
    ((local_time.dt.month == 12) & (local_time.dt.day >= 24))
    | ((local_time.dt.month == 1) & (local_time.dt.day <= 6))
).astype(int)

calendar.head()

,timestamp_utc,hour_local,day_of_week,month,year,is_weekend,is_holiday,is_summer,is_christmas_period
0,2022-01-01 00:00:00+00:00,1,5,1,2022,1,1,0,1
1,2022-01-01 01:00:00+00:00,2,5,1,2022,1,1,0,1
2,2022-01-01 02:00:00+00:00,3,5,1,2022,1,1,0,1
3,2022-01-01 03:00:00+00:00,4,5,1,2022,1,1,0,1
4,2022-01-01 04:00:00+00:00,5,5,1,2022,1,1,0,1


##Checking validity

Checking: row count, missing values, unique UTC timestamps, hourly spacing, calendar features for selected known dates.

In [44]:

assert len(calendar) == 35064
assert calendar["timestamp_utc"].is_unique
assert not calendar.isna().any().any()


gaps = calendar["timestamp_utc"].diff().dropna()
assert gaps.eq(pd.Timedelta(hours=1)).all()


def check_hour(timestamp, expected):
    row = calendar.loc[
        calendar["timestamp_utc"] == pd.Timestamp(timestamp)
    ].iloc[0]

    for column, value in expected.items():
        assert row[column] == value, f"{timestamp}: check {column}"


check_hour("2022-01-01T00:00:00Z", {
    "hour_local": 1,
    "day_of_week": 5,
    "is_weekend": 1,
    "is_holiday": 1,
    "is_summer": 0,
    "is_christmas_period": 1
})


check_hour("2022-07-04T10:00:00Z", {
    "hour_local": 12,
    "day_of_week": 0,
    "is_weekend": 0,
    "is_holiday": 0,
    "is_summer": 1,
    "is_christmas_period": 0
})

print("All checks passed!")

All checks passed!


In [45]:
hours_per_day = calendar.groupby(local_time.dt.date).size()

full_days = hours_per_day.iloc[1:-1]

print(full_days[full_days != 24])

timestamp_utc
2022-03-27    23
2022-10-30    25
2023-03-26    23
2023-10-29    25
2024-03-31    23
2024-10-27    25
2025-03-30    23
2025-10-26    25
dtype: int64


In [46]:
from pathlib import Path

In [47]:
repo_root = Path.cwd()

while not (repo_root / ".git").exists():
    if repo_root == repo_root.parent:
        raise FileNotFoundError("Notebook must be inside the repository.")
    repo_root = repo_root.parent



In [48]:
output_path = repo_root / "data" / "processed" / "calendar_features.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)



In [49]:
calendar.to_csv(
    output_path,
    index=False,
    date_format="%Y-%m-%dT%H:%M:%SZ"
)



##Definitions and checks

Period: 2022-01-01 00:00 UTC through 2025-12-31 23:00 UTC.
One row per UTC hour.
Calendar features use Italian local time: Europe/Rome.
Monday = 0, Sunday = 6.
Weekend: Saturday and Sunday.
Public holidays: Italian national holidays from the holidays package.
Summer: June–August (provisional).
Christmas period: December 24–January 6 inclusive (provisional).
Date range and provisional definitions require team confirmation.
Optional daylight_hours awaits team confirmation.

Validation checks cover row count, missing values, unique timestamps, hourly spacing, selected known dates and daylight saving changes.